# 01 — Feature Engineering

Per-player rolling features are pre-computed in the ClickHouse ETL.
This notebook handles: bio embeddings, H2H, pairwise construction,
train/test split, and LSTM sequence table.

In [ ]:
gold_table = "gold.match_features"
profiles_table = "gold.player_profiles"
test_size = 0.2
random_state = 42
cutoff_date = "2025-06-01"
output_dir = "data/processed"

In [ ]:
import json
import mlflow
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
from src.db.client import get_client, to_dataframe
from src.features.rolling import (
    load_bio_embeddings,
    join_bio_embeddings,
    compute_h2h_features,
    build_pairwise,
    FEATURE_COLS,
    SEQ_FEATURE_COLS,
)

client = get_client()
Path(output_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
print("Loading gold features...")
df = to_dataframe(f"SELECT * FROM {gold_table} ORDER BY match_date", client)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Date range: {df['match_date'].min()} to {df['match_date'].max()}")

In [ ]:
print("Computing bio embeddings...")
profiles = to_dataframe(f"SELECT player_id, summary FROM {profiles_table}", client)
bio_df = load_bio_embeddings(profiles)
df = join_bio_embeddings(df, bio_df)
print(f"Bio dim: {len([c for c in df.columns if c.startswith('bio_')])}")

In [ ]:
print("Computing H2H features...")
df = compute_h2h_features(df)
print("H2H done")

In [ ]:
print("Building pairwise matches...")
paired = build_pairwise(df)
print(f"Paired: {len(paired)} matches")

In [ ]:
# ── Build LSTM sequence table ──
print("Building sequence table...")
seq_cols = [*["player_id", "match_date", "match_id"], *SEQ_FEATURE_COLS]
sequences = df[seq_cols].sort_values(["player_id", "match_date"])
sequences.to_parquet(f"{output_dir}/sequences.parquet")
print(f"Sequences saved: {len(sequences)} rows")

In [ ]:
train = paired[paired["match_date"] < cutoff_date]
test = paired[paired["match_date"] >= cutoff_date]

X_train, y_train = train[FEATURE_COLS].copy(), train["match_won"]
X_test, y_test = test[FEATURE_COLS].copy(), test["match_won"]
info_cols = ["match_id", "match_date", "player_id", "opponent_id", "tournament", "round", "surface"]

# Impute NULL rolling features (fit on train only, no leakage)
mean_cols = [c for c in FEATURE_COLS if "rank" not in c and "streak" not in c]
median_cols = [c for c in FEATURE_COLS if "avg_opp_rank" in c or "streak" in c]
if mean_cols:
    mean_imputer = SimpleImputer(strategy="mean")
    X_train[mean_cols] = mean_imputer.fit_transform(X_train[mean_cols])
    X_test[mean_cols] = mean_imputer.transform(X_test[mean_cols])
if median_cols:
    median_imputer = SimpleImputer(strategy="median")
    X_train[median_cols] = median_imputer.fit_transform(X_train[median_cols])
    X_test[median_cols] = median_imputer.transform(X_test[median_cols])

In [ ]:
# ── Save ──
pd.DataFrame({"y": y_train}).to_parquet(f"{output_dir}/y_train.parquet", index=False)
pd.DataFrame({"y": y_test}).to_parquet(f"{output_dir}/y_test.parquet", index=False)

for name, data in [
    ("X_train", X_train),
    ("X_test", X_test),
    ("info_train", train[info_cols]),
    ("info_test", test[info_cols]),
    ("sequences", sequences),
]:
    path = f"{output_dir}/{name}.parquet"
    data.to_parquet(path)
    print(f"Saved {path} ({len(data)} rows)")

In [ ]:
with open(f"{output_dir}/feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)
print(f"{len(FEATURE_COLS)} features saved")

In [ ]:
mlflow.set_experiment("feature_engineering")
with mlflow.start_run():
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("cutoff_date", str(cutoff_date))
    mlflow.log_text("\n".join(FEATURE_COLS), "features.txt")
print("Done.")